# 03 — Diffusion generation

Loads the fine-tuned LoRA weights and samples a couple of images per category for qualitative inspection. Requires a CUDA GPU for sensible speed; CPU works for a single inference but is slow.

In [ ]:
from cv_diffusion.models.diffusion import DiffusionConfig, GenerationParams, StableDiffusionWrapper
from cv_diffusion.preprocessing.dataset import DISASTER_PROMPT_TEMPLATES, build_text_prompt

config = DiffusionConfig(
    model_id='runwayml/stable-diffusion-v1-5',
    torch_dtype='float16',
    lora_weights='../outputs/lora/sd15_disaster',  # set to None for the base model
    lora_scale=1.0,
)
wrapper = StableDiffusionWrapper(config)
print('Loaded.')

In [ ]:
import matplotlib.pyplot as plt
categories = list(DISASTER_PROMPT_TEMPLATES.keys())
fig, axes = plt.subplots(len(categories), 2, figsize=(8, 4*len(categories)))
for row, cat in enumerate(categories):
    prompts = [build_text_prompt(cat, index=i) for i in range(2)]
    images = wrapper.generate(GenerationParams(
        prompt=prompts,
        num_inference_steps=30,
        guidance_scale=7.5,
        height=512, width=512,
        seed=42 + row,
        scheduler='dpm++_2m',
    ))
    for col, img in enumerate(images):
        axes[row, col].imshow(img); axes[row, col].set_axis_off()
        axes[row, col].set_title(cat if col == 0 else '')
plt.tight_layout(); plt.show()